# Step 2: Cloud-Proof Masking — Sentinel-1 VH/VV Radar Fusion

Radar sees through cloud. The VH backscatter (dB) plus the VH/VV ratio separate forest from non-forest even on a fully overcast day. Synthetic linear-power backscatter and Sentinel-2 reflectance are generated here, then fused by `src/sar_optical_fusion.py`.

In [ ]:
import os
import sys

import matplotlib.pyplot as plt
import numpy as np
import rasterio
from rasterio.transform import from_bounds

sys.path.append(os.path.abspath('../src'))
from sar_optical_fusion import apply_cloud_masking, extract_ndvi_max

os.makedirs('../data/input', exist_ok=True)
os.makedirs('../data/output', exist_ok=True)

rng = np.random.default_rng(7)
tr = from_bounds(-6.4, 52.95, -6.1, 53.15, 200, 200)


def write(path, arr, dtype):
    with rasterio.open(path, 'w', driver='GTiff', height=200, width=200, count=1,
                       dtype=dtype, crs='EPSG:4326', transform=tr) as dst:
        dst.write(arr.astype(dtype), 1)


# Sentinel-1 backscatter in linear power (VH ~ -17..-12 dB, VV noisier)
write('../data/input/demo_s1_vh.tif', 0.02 + rng.random((200, 200)) * 0.06, 'float32')
write('../data/input/demo_s1_vv.tif', 0.08 + rng.random((200, 200)) * 0.15, 'float32')
# Sentinel-2 reflectance scaled x10000
write('../data/input/demo_s2_nir.tif', rng.random((200, 200)) * 4000 + 1000, 'uint16')
write('../data/input/demo_s2_red.tif', rng.random((200, 200)) * 2500 + 200, 'uint16')

mask_path = apply_cloud_masking('../data/input/demo_s1_vh.tif', '../data/input/demo_s1_vv.tif',
                                '../data/output/02_forest_mask.tif')
ndvi_path = extract_ndvi_max('../data/input/demo_s2_nir.tif', '../data/input/demo_s2_red.tif',
                             '../data/output/03_ndvi.tif')

fig, axes = plt.subplots(1, 2, figsize=(11, 4.5))
with rasterio.open(mask_path) as m:
    axes[0].imshow(m.read(1), cmap='Greens')
    axes[0].set_title('Sentinel-1 VH/VV forest mask (cloud-proof)')
with rasterio.open(ndvi_path) as n:
    im = axes[1].imshow(n.read(1), cmap='RdYlGn', vmin=0, vmax=1)
    axes[1].set_title('NDVI max composite')
fig.colorbar(im, ax=axes[1], fraction=0.04)
for ax in axes:
    ax.axis('off')
plt.show()

with rasterio.open(mask_path) as m:
    print(f"✅ Forest pixels: {int(m.read(1).sum())} / 40,000")